In [32]:
import pandas as pd
import numpy as np

df = pd.read_csv("../datasets/processed/canonical_dataset.csv")

In [33]:
df.head()

,triage_target,esi,chief_complaint,age,gender,race,lang,insurance_status,bmi,previous_ed_visits,...,leg_swelling,palpitations,vision_changes,recent_injury,unconscious,unable_to_speak,severe_bleeding,seizure_active,cyanosis,severe_shortness_of_breath
0,Home Care,4,fall_trauma,40,2,8,1,4,26.5,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Home Care,4,fall_trauma,66,2,4,1,1,39.8,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Immediate,2,dizziness,66,2,4,1,1,39.8,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Immediate,2,headache,66,2,4,1,1,39.8,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Consult,3,vomiting,84,1,5,2,3,39.8,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [34]:
df.shape

(357218, 64)

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 357218 entries, 0 to 357217
Data columns (total 64 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   triage_target               357218 non-null  object 
 1   esi                         357218 non-null  int64  
 2   chief_complaint             357218 non-null  object 
 3   age                         357218 non-null  int64  
 4   gender                      357218 non-null  int64  
 5   race                        357218 non-null  int64  
 6   lang                        357218 non-null  int64  
 7   insurance_status            357218 non-null  int64  
 8   bmi                         357218 non-null  float64
 9   previous_ed_visits          357218 non-null  int64  
 10  previous_admissions         357218 non-null  int64  
 11  asthma                      357218 non-null  float64
 12  copd                        357218 non-null  float64
 13  diabetes      

In [36]:
df.isnull().sum()

triage_target                 0
esi                           0
chief_complaint               0
age                           0
gender                        0
                             ..
unable_to_speak               0
severe_bleeding               0
seizure_active                0
cyanosis                      0
severe_shortness_of_breath    0
Length: 64, dtype: int64

In [37]:
df.duplicated().sum()

np.int64(7502)

In [38]:
df = df.drop_duplicates()

In [39]:
df.shape

(349716, 64)

In [40]:
df["triage_target"].value_counts()

triage_target
Consult      172909
Immediate     92727
Home Care     84080
Name: count, dtype: int64

In [41]:
df["chief_complaint"].value_counts()

chief_complaint
fall_trauma            68610
abdominal_pain         66347
chest_pain             34004
shortness_of_breath    26469
back_pain              20733
stroke_symptoms        14518
urinary_symptoms       14281
headache               12824
dizziness              12741
fever                  11682
cough                  11594
rash                   10893
vomiting               10517
leg_swelling            8731
sore_throat             5216
syncope                 5118
palpitations            5034
seizure                 3815
diarrhea                3313
allergic_reaction       3276
Name: count, dtype: int64

In [42]:
df["chest_tightness"].value_counts()

chest_tightness
0.0    349716
Name: count, dtype: int64

In [43]:
df["radiating_pain"].value_counts()

radiating_pain
0.0    349716
Name: count, dtype: int64

In [44]:
df["gender"].value_counts()

gender
1    203371
2    146345
Name: count, dtype: int64

In [45]:
# 1. Map target string classes to integers (0, 1, 2)
target_map = {"Home Care": 0, "Consult": 1, "Immediate": 2}
y = df["triage_target"].map(target_map)

# 2. Features matrix (Drop target and ESI leakage)
X = df.drop(columns=["triage_target", "esi"])



In [46]:
cc_mapping = {cc: idx for idx, cc in enumerate(sorted(X["chief_complaint"].unique()))}
X["chief_complaint"] = X["chief_complaint"].map(cc_mapping)
## this give cc 0 to 19 
print(X["chief_complaint"])

0          7
1          7
2          6
3          9
4         19
          ..
357213     2
357214     4
357215    18
357216    10
357217    10
Name: chief_complaint, Length: 349716, dtype: int64


In [47]:
onset_map = {"not_specified": 0, "gradual": 1, "acute": 2, "sudden": 3}
progression_map = {"not_specified": 0, "better": 1, "same": 2, "worse": 3}
pain_map = {"not_specified": 0, "sharp": 1, "burning": 2, "crushing": 3, "stabbing": 4, "pressure": 5}

X["symptom_onset"] = X["symptom_onset"].str.lower().map(onset_map).fillna(0).astype(int)
X["progression"] = X["progression"].str.lower().map(progression_map).fillna(0).astype(int)
X["pain_type"] = X["pain_type"].str.lower().map(pain_map).fillna(0).astype(int)


In [48]:
for col in ["gender", "race", "lang", "insurance_status"]:
    if X[col].dtype == 'object':
        X[col] = X[col].astype('category').cat.codes


In [49]:
import json

# Convert numpy int64 keys and values to standard python types
cc_mapping_clean = {str(cc): int(idx) for cc, idx in cc_mapping.items()}

# Save chief complaint encoder
with open("chief_complaint_encoder.json", "w") as f:
    json.dump(cc_mapping_clean, f, indent=2)

# Save feature list in exact order
with open("feature_names.json", "w") as f:
    json.dump(list(X.columns), f, indent=2)

print("Saved chief_complaint_encoder.json and feature_names.json successfully!")


Saved chief_complaint_encoder.json and feature_names.json successfully!


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 349716 entries, 0 to 357217
Data columns (total 64 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   triage_target               349716 non-null  object 
 1   esi                         349716 non-null  int64  
 2   chief_complaint             349716 non-null  object 
 3   age                         349716 non-null  int64  
 4   gender                      349716 non-null  int64  
 5   race                        349716 non-null  int64  
 6   lang                        349716 non-null  int64  
 7   insurance_status            349716 non-null  int64  
 8   bmi                         349716 non-null  float64
 9   previous_ed_visits          349716 non-null  int64  
 10  previous_admissions         349716 non-null  int64  
 11  asthma                      349716 non-null  float64
 12  copd                        349716 non-null  float64
 13  diabetes           

In [51]:
from sklearn.model_selection import train_test_split

# 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")


X_train shape: (279772, 62)
X_test shape : (69944, 62)


In [52]:
# Check for non-numeric columns (Must print: [])
non_numeric = X_train.select_dtypes(include=['object']).columns
print("Remaining text columns:", list(non_numeric))

# Check for NULLs (Must print: 0)
print("Total NULL values:", X_train.isna().sum().sum())


Remaining text columns: []
Total NULL values: 0


In [53]:
%pip install xgboost


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Asus f15\Machine-Learning\env\Scripts\python.exe -m pip install --upgrade pip


In [54]:
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

model.fit(X_train, y_train)


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [55]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import pandas as pd
import numpy as np

# 1. Predict on the Test Set
y_pred = model.predict(X_test)

# Class names corresponding to target labels
target_names = ["Home Care (0)", "Consult (1)", "Immediate (2)"]

# 2. Overall Performance Metrics
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"🎯 Test Accuracy : {acc * 100:.2f}%")
print(f"🎯 Macro F1-Score : {macro_f1 * 100:.2f}%\n")

# 3. Detailed Classification Report (Precision, Recall, F1-score per class)
print("==================================================================")
print("                   📊 CLASSIFICATION REPORT                       ")
print("==================================================================")
print(classification_report(y_test, y_pred, target_names=target_names, digits=4))

# 4. Confusion Matrix
print("==================================================================")
print("                    🧩 CONFUSION MATRIX                           ")
print("==================================================================")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm, 
    index=[f"Actual: {t}" for t in target_names], 
    columns=[f"Pred: {t}" for t in target_names]
)
print(cm_df)

# 5. Top 15 Most Important Model Features
print("\n==================================================================")
print("                 ⭐ TOP 15 FEATURE IMPORTANCES                    ")
print("==================================================================")
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(15).round(4))


🎯 Test Accuracy : 68.19%
🎯 Macro F1-Score : 66.81%

                   📊 CLASSIFICATION REPORT                       
               precision    recall  f1-score   support

Home Care (0)     0.6778    0.7301    0.7030     16816
  Consult (1)     0.6829    0.7488    0.7143     34582
Immediate (2)     0.6846    0.5135    0.5869     18546

     accuracy                         0.6819     69944
    macro avg     0.6818    0.6642    0.6681     69944
 weighted avg     0.6821    0.6819    0.6778     69944

                    🧩 CONFUSION MATRIX                           
                       Pred: Home Care (0)  Pred: Consult (1)  \
Actual: Home Care (0)                12278               4286   
Actual: Consult (1)                   4552              25895   
Actual: Immediate (2)                 1284               7738   

                       Pred: Immediate (2)  
Actual: Home Care (0)                  252  
Actual: Consult (1)                   4135  
Actual: Immediate (2)           

In [56]:
import joblib
import json

# 1. Save the XGBoost model
joblib.dump(model, "triage_model.joblib")

# 2. Save the exact feature order (CRITICAL - must match your backend!)
feature_names = list(X_train.columns)
with open("feature_names.json", "w") as f:
    json.dump(feature_names, f, indent=2)

print("Saved triage_model.joblib and feature_names.json!")
print(f"Feature count: {len(feature_names)}")
print(feature_names)


Saved triage_model.joblib and feature_names.json!
Feature count: 62
['chief_complaint', 'age', 'gender', 'race', 'lang', 'insurance_status', 'bmi', 'previous_ed_visits', 'previous_admissions', 'asthma', 'copd', 'diabetes', 'hypertension', 'heart_disease', 'chf', 'kidney_disease', 'epilepsy', 'anemia', 'influenza', 'uti', 'dizziness', 'fatigue', 'allergy', 'takes_antihypertensive', 'takes_diabetes_medicine', 'takes_inhaler', 'takes_blood_thinner', 'heart_rate', 'systolic_bp', 'diastolic_bp', 'respiratory_rate', 'spo2', 'body_temperature', 'symptom_duration', 'symptom_severity', 'pain_score', 'pain_type', 'symptom_onset', 'progression', 'intermittent', 'relieved_by_rest', 'radiating_pain', 'chest_tightness', 'neck_stiffness', 'light_sensitivity', 'persistent_cough', 'bloody_cough', 'body_aches', 'abdominal_tenderness', 'diarrhea', 'burning_urination', 'urinary_frequency', 'leg_swelling', 'palpitations', 'vision_changes', 'recent_injury', 'unconscious', 'unable_to_speak', 'severe_bleeding